In [1]:
!pip install trl bitsandbytes>=0.46.1 -q

In [2]:
import torch
from datasets import Dataset, load_dataset
from transformers import (
 AutoTokenizer,
 AutoModelForCausalLM,
 BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer


In [3]:
MODEL_C = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer_c = AutoTokenizer.from_pretrained(MODEL_C)
if tokenizer_c.pad_token is None:
 tokenizer_c.pad_token = tokenizer_c.eos_token

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

In [4]:
use_qlora = torch.cuda.is_available()
if use_qlora:
 compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
 quant_config = BitsAndBytesConfig(
 load_in_4bit=True,
 bnb_4bit_quant_type="nf4",
 bnb_4bit_use_double_quant=True,
 bnb_4bit_compute_dtype=compute_dtype,
 )
 base_c = AutoModelForCausalLM.from_pretrained(
 MODEL_C,
 quantization_config=quant_config,
 device_map="auto",)

 base_c = prepare_model_for_kbit_training(base_c)
else:
 base_c = AutoModelForCausalLM.from_pretrained(MODEL_C)
lora_config = LoraConfig(
  task_type=TaskType.CAUSAL_LM,
  r=8,
  lora_alpha=16,
  lora_dropout=0.05,
  target_modules=["q_proj", "v_proj"],
  bias="none",
)
model_c = get_peft_model(base_c, lora_config)
model_c.print_trainable_parameters()


model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


In [5]:
sft_dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/support_specialist_sft_train_48.jsonl",
        "validation": "/content/support_specialist_sft_validation_12.jsonl",
    }
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [6]:
def format_for_sft(example):
 return {
 "text": tokenizer_c.apply_chat_template(
 example["messages"],
 tokenize=False,
 add_generation_prompt=False,
 )
 }
sft_train = sft_dataset["train"].map(format_for_sft)
sft_val = sft_dataset["validation"].map(format_for_sft)
sft_args = SFTConfig(
 output_dir="models/support_adapter",
 num_train_epochs=10,
 per_device_train_batch_size=2,
 per_device_eval_batch_size=2,
 learning_rate=2e-4,
 eval_strategy="epoch",
 save_strategy="epoch",
 load_best_model_at_end=True,
 metric_for_best_model="eval_loss",
 greater_is_better=False,
 dataset_text_field="text",
 max_length=512,
 packing=False,
 report_to="none",
)
trainer_c = SFTTrainer(
 model=model_c,
 args=sft_args,
 train_dataset=sft_train,
 eval_dataset=sft_val,
 processing_class=tokenizer_c,
)
# The fresh LoRA adapter is a no-op before training; this measures the base model.
import math
baseline_c = trainer_c.evaluate(metric_key_prefix="baseline")
baseline_c["baseline_perplexity"] = math.exp(baseline_c["baseline_loss"]) if baseline_c["baseline_loss"] < 20 else float("inf")
print("BASELINE C (validation):", baseline_c)
trainer_c.train()
fine_tuned_c = trainer_c.evaluate(metric_key_prefix="fine_tuned")
fine_tuned_c["fine_tuned_perplexity"] = math.exp(fine_tuned_c["fine_tuned_loss"]) if fine_tuned_c["fine_tuned_loss"] < 20 else float("inf")
print("FINE-TUNED C (same validation):", fine_tuned_c)
import pandas as pd
history_c = pd.DataFrame(trainer_c.state.log_history)
train_logs_c = history_c[history_c["loss"].notna()].copy() if "loss" in history_c else pd.DataFrame()
eval_logs_c = history_c[history_c["eval_loss"].notna()].copy() if "eval_loss" in history_c else pd.DataFrame()
print("Real trainer log rows:", len(train_logs_c), "train /", len(eval_logs_c), "eval")
model_c.save_pretrained("models/support_adapter")
tokenizer_c.save_pretrained("models/support_adapter")

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

...

BASELINE C (validation): {'baseline_loss': 4.017305374145508, 'eval_entropy': 3.5356472730636597, 'eval_num_tokens': 0.0, 'eval_mean_token_accuracy': 0.2819748868544896, 'baseline_perplexity': 55.551214228496}


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,3.860018,3.580231,3.487224,6779.000000,0.364682
2,3.423063,3.008347,3.005267,13558.000000,0.451604
3,2.725224,2.479210,2.669252,20337.000000,0.567708
4,2.263592,2.164088,2.281306,27116.000000,0.612808
5,2.084667,2.076009,2.122844,33895.000000,0.633099
6,2.071620,2.046522,2.091287,40674.000000,0.633739
7,2.045488,2.033024,2.079831,47453.000000,0.633260
8,2.029156,2.024242,2.065700,54232.000000,0.633260
9,2.070835,2.023114,2.062874,61011.000000,0.633855
10,2.068589,2.022227,2.062228,67790.000000,0.633855


Training Loss,Validation Loss,Epoch,Tuned Loss,Entropy,Num Tokens,Mean Token Accuracy
2.068589,No log,10,2.022227,2.062228,67790.000000,0.633855


FINE-TUNED C (same validation): {'fine_tuned_loss': 2.0222268104553223, 'eval_entropy': 2.0622281034787497, 'eval_num_tokens': 67790.0, 'eval_mean_token_accuracy': 0.633854866027832, 'fine_tuned_perplexity': 7.555130058409286}
Real trainer log rows: 24 train / 10 eval


('models/support_adapter/tokenizer_config.json',
 'models/support_adapter/chat_template.jinja',
 'models/support_adapter/tokenizer.json')